## Sprint 10 Project

In [6]:
#Import libraries and functions

import pandas as pd #Import pandas
import numpy as np #Import numpy
from IPython.display import Image, display, HTML #For image viewing and accessible formatting
import io #For formatting
from sklearn.model_selection import train_test_split #For model training
from sklearn.linear_model import LinearRegression #For model training
from numpy.random import RandomState #Import RandomState for bootstrapping
from sklearn.metrics import root_mean_squared_error

### i. Introduction

As of 2024, the global gold mining market was estimated at $260.86 billion, and is estimated to grow to $710.08 billion by 2033, according to Grand View Research (https://www.grandviewresearch.com/industry-analysis/gold-mining-market-report). 

In [7]:
mining_url = "https://raw.githubusercontent.com/DHE42/sprint_10_repository/main/image_4.png"
display(Image(url=mining_url))

The industry relies on a two part refinement process composed of **flotation** and **purification**. During flotation, a substance called **flotation pulp** containing **reagent additions** is introduced into the **rougher feed**, or raw gold ore, to separate **concentrate** from **tails**. Concentrate is the ore in various stages of purification, and tails are residues from the refinement process that contain a low amount of valuable metals. The goal within the overall refinement process is to transform the rougher feed into a **final Au (gold) concentrate** that will have the highest possible amount of gold, and to produce **rougher tails** that have the lowest possible amount of valuable metals in them, as tails are discarded at the end of the refinement process.



In [8]:
refinement_url = "https://raw.githubusercontent.com/DHE42/sprint_10_repository/main/Image%206.png"
display(Image(url=refinement_url))

The goal of refinement in the gold industry is to maximize market product and minimize waste at every step. Data scientists are useful for this since they can use sound mathematical computing to predict recovery rates. **Recovery rates** are the percentage of gold successfully extracted from rougher feed at flotation to products produced through purification into the final Au concentrate and tails. Data scientists enable companies to sell their refined gold products at the highest possible market value by successfully predicting recovery rates to optimize extraction efficiency and maximize profitability.

In [9]:
profitability_url = "https://raw.githubusercontent.com/DHE42/sprint_10_repository/main/maximum_profitability.png"
display(Image(url=profitability_url))

### 1. Prepare the Data

#### 1.1 Open the files and look into the data.

In [10]:
#Read and name CSVs

def read_and_display_csvs(url_list, df_names=None, head_rows=5):
    """
    Reads multiple CSVs from URLs into pandas DataFrames, optionally names them,
    and displays their head and info side by side in Jupyter.
    
    Parameters:
        url_list (list): List of CSV URLs to read.
        df_names (list or None): Optional list of names for each DataFrame.
        head_rows (int): Number of rows to show for the head of each DataFrame.
        
    Returns:
        dict or list: Dictionary of DataFrames keyed by df_names if provided, else list of DataFrames.
    """
    dfs = []
    for url in url_list:
        dfs.append(pd.read_csv(url))
    
    # Assign names if provided
    if df_names:
        dfs_dict = {name: df for name, df in zip(df_names, dfs)}
        df_list_for_display = dfs_dict.values()
    else:
        dfs_dict = dfs
        df_list_for_display = dfs
    
    # Generate default names for display if not provided
    display_names = df_names if df_names else [f"DF_{i+1}" for i in range(len(dfs))]
    
    # Display head for each DataFrame
    for name, df in zip(display_names, df_list_for_display):
        display(HTML(f"<h4>{name} Head</h4>"))
        display(df.head(head_rows))
    
    # Build side-by-side info display
    html = "<div style='display:flex; flex-wrap:wrap;'>"
    for name, df in zip(display_names, df_list_for_display):
        buf = io.StringIO()
        df.info(buf=buf)
        html += (
            f"<pre style='margin-right:30px; width:300px; overflow:auto;'>"
            f"<b>{name} Info</b>\n{buf.getvalue()}</pre>"
        )
    html += "</div>"
    
    display(HTML(f"<h4>DataFrame Info</h4>{html}"))
    
    return dfs_dict

urls = [
    "https://raw.githubusercontent.com/DHE42/sprint_10_repository/main/gold_recovery_full.csv",
    "https://raw.githubusercontent.com/DHE42/sprint_10_repository/main/gold_recovery_train.csv",
    "https://raw.githubusercontent.com/DHE42/sprint_10_repository/main/gold_recovery_test.csv"
]

names = ["full_df", "train_df", "test_df"]

dfs = read_and_display_csvs(urls, df_names=names)

# Access individual DataFrames
df_1 = dfs["full_df"]
df_2 = dfs["train_df"]
df_3 = dfs["test_df"]

,date,final.output.concentrate_ag,final.output.concentrate_pb,final.output.concentrate_sol,final.output.concentrate_au,final.output.recovery,final.output.tail_ag,final.output.tail_pb,final.output.tail_sol,final.output.tail_au,...,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
0,2016-01-15 00:00:00,6.055403,9.889648,5.507324,42.192020,70.541216,10.411962,0.895447,16.904297,2.143149,...,14.016835,-502.488007,12.099931,-504.715942,9.925633,-498.310211,8.079666,-500.470978,14.151341,-605.841980
1,2016-01-15 01:00:00,6.029369,9.968944,5.257781,42.701629,69.266198,10.462676,0.927452,16.634514,2.224930,...,13.992281,-505.503262,11.950531,-501.331529,10.039245,-500.169983,7.984757,-500.582168,13.998353,-599.787184
2,2016-01-15 02:00:00,6.055926,10.213995,5.383759,42.657501,68.116445,10.507046,0.953716,16.208849,2.257889,...,14.015015,-502.520901,11.912783,-501.133383,10.070913,-500.129135,8.013877,-500.517572,14.028663,-601.427363
3,2016-01-15 03:00:00,6.047977,9.977019,4.858634,42.689819,68.347543,10.422762,0.883763,16.532835,2.146849,...,14.036510,-500.857308,11.999550,-501.193686,9.970366,-499.201640,7.977324,-500.255908,14.005551,-599.996129
4,2016-01-15 04:00:00,6.148599,10.142511,4.939416,42.774141,66.927016,10.360302,0.792826,16.525686,2.055292,...,14.027298,-499.838632,11.953070,-501.053894,9.925709,-501.686727,7.894242,-500.356035,13.996647,-601.496691


,date,final.output.concentrate_ag,final.output.concentrate_pb,final.output.concentrate_sol,final.output.concentrate_au,final.output.recovery,final.output.tail_ag,final.output.tail_pb,final.output.tail_sol,final.output.tail_au,...,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
0,2016-01-15 00:00:00,6.055403,9.889648,5.507324,42.192020,70.541216,10.411962,0.895447,16.904297,2.143149,...,14.016835,-502.488007,12.099931,-504.715942,9.925633,-498.310211,8.079666,-500.470978,14.151341,-605.841980
1,2016-01-15 01:00:00,6.029369,9.968944,5.257781,42.701629,69.266198,10.462676,0.927452,16.634514,2.224930,...,13.992281,-505.503262,11.950531,-501.331529,10.039245,-500.169983,7.984757,-500.582168,13.998353,-599.787184
2,2016-01-15 02:00:00,6.055926,10.213995,5.383759,42.657501,68.116445,10.507046,0.953716,16.208849,2.257889,...,14.015015,-502.520901,11.912783,-501.133383,10.070913,-500.129135,8.013877,-500.517572,14.028663,-601.427363
3,2016-01-15 03:00:00,6.047977,9.977019,4.858634,42.689819,68.347543,10.422762,0.883763,16.532835,2.146849,...,14.036510,-500.857308,11.999550,-501.193686,9.970366,-499.201640,7.977324,-500.255908,14.005551,-599.996129
4,2016-01-15 04:00:00,6.148599,10.142511,4.939416,42.774141,66.927016,10.360302,0.792826,16.525686,2.055292,...,14.027298,-499.838632,11.953070,-501.053894,9.925709,-501.686727,7.894242,-500.356035,13.996647,-601.496691


,date,primary_cleaner.input.sulfate,primary_cleaner.input.depressant,primary_cleaner.input.feed_size,primary_cleaner.input.xanthate,primary_cleaner.state.floatbank8_a_air,primary_cleaner.state.floatbank8_a_level,primary_cleaner.state.floatbank8_b_air,primary_cleaner.state.floatbank8_b_level,primary_cleaner.state.floatbank8_c_air,...,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
0,2016-09-01 00:59:59,210.800909,14.993118,8.080000,1.005021,1398.981301,-500.225577,1399.144926,-499.919735,1400.102998,...,12.023554,-497.795834,8.016656,-501.289139,7.946562,-432.317850,4.872511,-500.037437,26.705889,-499.709414
1,2016-09-01 01:59:59,215.392455,14.987471,8.080000,0.990469,1398.777912,-500.057435,1398.055362,-499.778182,1396.151033,...,12.058140,-498.695773,8.130979,-499.634209,7.958270,-525.839648,4.878850,-500.162375,25.019940,-499.819438
2,2016-09-01 02:59:59,215.259946,12.884934,7.786667,0.996043,1398.493666,-500.868360,1398.860436,-499.764529,1398.075709,...,11.962366,-498.767484,8.096893,-500.827423,8.071056,-500.801673,4.905125,-499.828510,24.994862,-500.622559
3,2016-09-01 03:59:59,215.336236,12.006805,7.640000,0.863514,1399.618111,-498.863574,1397.440120,-499.211024,1400.129303,...,12.033091,-498.350935,8.074946,-499.474407,7.897085,-500.868509,4.931400,-499.963623,24.948919,-498.709987
4,2016-09-01 04:59:59,199.099327,10.682530,7.530000,0.805575,1401.268123,-500.808305,1398.128818,-499.504543,1402.172226,...,12.025367,-500.786497,8.054678,-500.397500,8.107890,-509.526725,4.957674,-500.360026,25.003331,-500.856333


#### 1.2 Check that recovery is calculated correctly. Using the training set, calculate recovery for the rougher.output.recovery feature. Find the MAE between your calculations and the feature values. Provide findings.

#### 1.3. Analyze the features not available in the test set. What are these parameters? What is their type?

#### 1.4. Perform data preprocessing.

### 2. Analyze the Data

#### 2.1. Take note of how the concentrations of metals (Au, Ag, Pb) change depending on the purification stage.

#### 2.2. Compare the feed particle size distributions in the training set and in the test set. If the distributions vary significantly, the model evaluation will be incorrect.

#### 2.3. Consider the total concentrations of all substances at different stages: raw feed, rougher concentrate, and final concentrate. Do you notice any abnormal values in the total distribution? If you do, is it worth removing such values from both samples? Describe the findings and eliminate anomalies. 

### 3. Build the Model

#### 3.1. Write a function to calculate the final sMAPE value.

#### 3.2. Train different models. Evaluate them using cross-validation. Pick the best model and test it using the test sample. Provide findings.

Use these formulae for evaluation metrics:

In [11]:
#URLs for evaluation metric formula images
smape_url = "https://raw.githubusercontent.com/DHE42/sprint_10_repository/89e3644270ee88604f5d53b8935bfd0a777352cd/Image.png"
final_smape_url = "https://raw.githubusercontent.com/DHE42/sprint_10_repository/89e3644270ee88604f5d53b8935bfd0a777352cd/Image%201.png"

In [12]:
#Display url_1
display(Image(url=smape_url))
print()

In [13]:
#Display url_2
display(Image(url=final_smape_url))
print()

### 4. Self Evaluation Before Submission

Answer each of the questions below to check your reasoning so you can ensure this project is the best you can produce.

#### 4.1 Have you prepared and analyzed the data properly?

#### 4.2 What models have you developed?

#### 4.3 How did you check the model‘s quality?

#### 4.4 Have you followed all the steps of the instructions?

#### 4.5 Did you keep to the project structure and explain the steps performed?

#### 4.6 What are your findings?

#### 4.7 Have you kept the code neat and avoided code duplication?

### 5. Conclusion

Write your conclusion based upon the self evaluation.